# attach_returns_acc_methods
用于装饰类：
```python
@attach_returns_acc_methods(config)
class A(Portfolio): ...
# 相当于 attach_returns_acc_methods(config)(A)
```
遍历 `config: Config` 中的每一项 (target_name, settings)
- 设置 source_name：从 settings 中获取对应键 source_name 的值，否则用 target_name
- 为类 `A` 创建一个名为 target_name 的方法 `cached_method(new_method)`，该方法有一个默认参数 _source_name: str = source_name
- 调用该方法时
    - 首先构造收益率访问器 `ReturnsSR/DFAccessor` 对象
    - 接着获取收益率访问器对象的名为 _source_name 的方法并调用

## 源码
```python
def attach_returns_acc_methods(config: Config) -> WrapperFuncT:
    def wrapper(cls: tp.Type[tp.T]) -> tp.Type[tp.T]:
        checks.assert_subclass_of(cls, "Portfolio")

        for target_name, settings in config.items():
            source_name = settings.get('source_name', target_name)
            docstring = settings.get('docstring', f"See `vectorbt.returns.accessors.ReturnsAccessor.{source_name}`.")

            def new_method(self,
                           *,
                           group_by: tp.GroupByLike = None,
                           benchmark_rets: tp.Optional[tp.ArrayLike] = None,
                           freq: tp.Optional[tp.FrequencyLike] = None,
                           year_freq: tp.Optional[tp.FrequencyLike] = None,
                           use_asset_returns: bool = False,
                           _source_name: str = source_name,
                           **kwargs) -> tp.Any:
                returns_acc = self.get_returns_acc(
                    group_by=group_by,
                    benchmark_rets=benchmark_rets,
                    freq=freq,
                    year_freq=year_freq,
                    use_asset_returns=use_asset_returns
                )
                return getattr(returns_acc, _source_name)(**kwargs)

            new_method.__name__ = target_name
            new_method.__qualname__ = f"{cls.__name__}.{target_name}"
            new_method.__doc__ = docstring
            setattr(cls, target_name, cached_method(new_method))
        return cls

    return wrapper
```